# 01 — Treinar UM modelo (template)

Escolha o `CONFIG` na célula de parâmetros e execute. Os artefatos são
salvos em `experiments/<modelo>__<timestamp>__<runid>/` no Drive.

## 1. Montar o Google Drive e localizar o projeto

⚠️ **Suba o PROJETO INTEIRO para o Drive**, não só a pasta `Data`. A pasta
no Drive (por padrão `/content/drive/MyDrive/Fato_Tese_Ruido`) precisa conter:

```
Fato_Tese_Ruido/
├── requirements.txt
├── src/ftr/...        ← o módulo 'ftr' (sem isto: No module named 'ftr')
├── configs/
├── scripts/
└── Data/dataset_FINAL.json
```

A célula abaixo **localiza a pasta automaticamente** (procurando por
`src/ftr`) e mostra um checklist do que está presente/faltando.

In [ ]:
import os, sys, glob

# Monta o Drive (somente no Colab).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Não está no Colab — usando caminho local.')


def _looks_like_project(p):
    return os.path.isdir(os.path.join(p, 'src', 'ftr'))


def _autodetect_root():
    candidates = ['/content/drive/MyDrive/Fato_Tese_Ruido']
    # procura qualquer pasta com src/ftr dentro do MyDrive (2 níveis)
    for base in ('/content/drive/MyDrive', '/content/drive/My Drive'):
        candidates += glob.glob(os.path.join(base, '**', 'src', 'ftr'), recursive=False)
        candidates += [os.path.dirname(os.path.dirname(p))
                       for p in glob.glob(os.path.join(base, '*', 'src', 'ftr'))]
    for c in candidates:
        if c and _looks_like_project(c):
            return c
    return candidates[0] if candidates else os.getcwd()


PROJECT_ROOT = _autodetect_root() if IN_COLAB else os.getcwd()
os.environ['FTR_PROJECT_ROOT'] = PROJECT_ROOT
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)

# Itens OBRIGATÓRIOS (o código + os dados).
required = {'src/ftr': 'pasta (módulo ftr)', 'configs': 'pasta',
            'scripts': 'pasta', 'Data/dataset_FINAL.json': 'dataset'}
# Itens OPCIONAIS (o Drive às vezes converte .txt/.xlsx p/ Google Docs/Sheets,
# removendo a extensão — sem problema, instalamos as deps explicitamente abaixo).
optional = {'requirements.txt': 'arquivo (opcional)'}
print('\nChecklist da pasta do projeto:')
missing = []
for rel, kind in required.items():
    ok = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    print(f'  [{"OK" if ok else "FALTA"}] {rel}  ({kind})')
    if not ok:
        missing.append(rel)
for rel, kind in optional.items():
    ok = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    print(f'  [{"OK" if ok else "--"}] {rel}  ({kind})')
if missing:
    raise SystemExit('\nFALTAM itens OBRIGATÓRIOS acima. Suba a PASTA INTEIRA do '
                     'projeto para o Drive (não só Data/).')
print('\nOK — itens obrigatórios presentes. (requirements.txt é opcional.)')

## 2. Instalar dependências

Instala as bibliotecas explicitamente (não depende do `requirements.txt`, que o
Drive às vezes converte para Google Docs). torch já vem no Colab.

In [ ]:
if IN_COLAB:
    !pip install -q 'transformers==4.48.3' 'datasets==3.3.0' 'scikit-learn==1.6.1' \
        evaluate sentencepiece sacremoses 'accelerate>=1.3.0' openpyxl seaborn pyarrow
    print('\nDependências instaladas.')
else:
    print('Local: instale com  pip install -r requirements.txt')

## 3. Verificar GPU, ambiente e caminhos

In [ ]:
import torch
from ftr.env import describe_environment, get_paths
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
import json
print(json.dumps(describe_environment(), indent=2, ensure_ascii=False))

## 4. Verificar que o dataset está no lugar

In [ ]:
paths = get_paths()
print('Dataset bruto:', paths.raw_dataset)
assert paths.raw_dataset.exists(), (
    f'Dataset não encontrado em {paths.raw_dataset}. '
    'Suba Data/dataset_FINAL.json para o Drive.')
print('OK — dataset encontrado (%.1f MB).' % (paths.raw_dataset.stat().st_size/1e6))

## 5. Parâmetros do experimento

Configs disponíveis: `roberta_lexpt`, `roberta_crawlpt`, `bertimbau_large`,
`mbert`, `bertimbau_base`, `xlmr`, `mdeberta_v3`, `longformer_pt`.

In [ ]:
CONFIG = 'configs/roberta_lexpt.yaml'  # @param
SMOKE_TEST = False  # @param  — True faz um teste rápido (5 passos)
max_steps = 5 if SMOKE_TEST else None

## 6. Preparar dados (se ainda não preparou)

Inclui split **por documento** (anti-leakage). Se já rodou no `02_run_all_colab.ipynb`
pode pular esta célula — os splits já existem.

In [ ]:
from ftr.data import load_splits
splits = load_splits(paths.data_processed)
if 'train' not in splits:
    print('Preparando dados (split por documento)...')
    !cd "{PROJECT_ROOT}" && python -W all scripts/01_prepare_data.py --config {CONFIG}
    splits = load_splits(paths.data_processed)
print('Splits:', {k: len(v) for k, v in splits.items()})

## 7. Treinar

In [ ]:
from ftr.config import load_config
from ftr.train import train_experiment
cfg = load_config(CONFIG, configs_dir=paths.configs)
status = train_experiment(cfg, paths, splits=splits, max_steps=max_steps)
print('Status:', status['status'], '| dir:', status['run_dir'])

## 8. Ver métricas e gráficos do experimento

In [ ]:
import json, glob
from IPython.display import Image, display
run_dir = paths.experiments / status['run_dir']
with open(run_dir / 'metrics' / 'test_metrics.json', encoding='utf-8') as f:
    m = json.load(f)
print('Accuracy :', round(m['accuracy'], 4))
print('F1-macro :', round(m['f1_macro'], 4))
print('MCC      :', round(m['mcc'], 4))
for fig in sorted(glob.glob(str(run_dir / 'figures' / '*_pt.png'))):
    display(Image(fig))